# 02 EDA Job-Level (2025)

This notebook performs reproducible EDA using only job-level datasets:
- `outputs/master_2025_joblevel_submission.csv.gz`
- `outputs/master_2025_joblevel_terminal.csv.gz`

It avoids loading `raw_valid` into memory.


## 1. Imports & Paths

In [ ]:
import os
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("ggplot")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "environment.yml").exists() and (PROJECT_ROOT.parent / "environment.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SUB_PATH = OUTPUT_DIR / "master_2025_joblevel_submission.csv.gz"
TER_PATH = OUTPUT_DIR / "master_2025_joblevel_terminal.csv.gz"
FINDINGS_PATH = OUTPUT_DIR / "eda_key_findings.md"

assert SUB_PATH.exists(), f"Missing file: {SUB_PATH}"
assert TER_PATH.exists(), f"Missing file: {TER_PATH}"

print(f"PROJECT_ROOT: {PROJECT_ROOT.resolve()}")
print(f"SUB_PATH: {SUB_PATH}")
print(f"TER_PATH: {TER_PATH}")
print(f"FIG_DIR: {FIG_DIR}")

FIGURE_FILES = []
METRICS = {}


def safe_to_datetime(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series.astype("string"), errors="coerce")


def safe_to_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype("string"), errors="coerce")


def get_col(df: pd.DataFrame, col: str, dtype: str = "string") -> pd.Series:
    if col in df.columns:
        return df[col]
    return pd.Series(pd.NA, index=df.index, dtype=dtype)


def save_current_fig(name: str):
    out = FIG_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(out, dpi=160, bbox_inches="tight")
    FIGURE_FILES.append(str(out))
    print(f"Saved figure: {out}")


def pct(x: float) -> str:
    if x is None or pd.isna(x):
        return "N/A"
    return f"{100*x:.2f}%"


def topn_with_other(series: pd.Series, n: int = 10) -> pd.Series:
    s = series.astype("string").str.strip()
    s = s.mask(s.isna() | s.eq(""), "MISSING")
    vc = s.value_counts(dropna=False)
    top = vc.head(n).copy()
    other = int(vc.iloc[n:].sum()) if len(vc) > n else 0
    if other > 0:
        top.loc["Other"] = other
    return top


def summary_quantiles_hours(seconds_series: pd.Series) -> dict:
    s = safe_to_numeric(seconds_series)
    s = s.replace([np.inf, -np.inf], np.nan)
    s = s[s.notna() & (s >= 0)] / 3600.0
    if s.empty:
        return {"count": 0, "median": np.nan, "p90": np.nan, "p95": np.nan, "p99": np.nan}
    return {
        "count": int(s.shape[0]),
        "median": float(s.quantile(0.50)),
        "p90": float(s.quantile(0.90)),
        "p95": float(s.quantile(0.95)),
        "p99": float(s.quantile(0.99)),
    }


## 2. Load job-level datasets (submission / terminal)

In [ ]:
sub = pd.read_csv(SUB_PATH, compression="gzip", dtype="string")
ter = pd.read_csv(TER_PATH, compression="gzip", dtype="string")

print("submission shape:", sub.shape)
print("terminal shape:  ", ter.shape)
print("\nsubmission columns:")
print(sub.columns.tolist())
print("\nterminal columns:")
print(ter.columns.tolist())

# Key type conversions
for df in (sub, ter):
    for c in ["Submit", "Eligible", "Start", "End"]:
        if c in df.columns:
            df[c] = safe_to_datetime(df[c])

for df in (sub, ter):
    for c in ["WaitTimeSec", "RunTimeSec", "ReqCPUS", "ReqMem_MB", "ReqNodes", "AllocCPUS", "AllocNodes", "Priority"]:
        if c in df.columns:
            df[c] = safe_to_numeric(df[c])

print("\nHead (submission)")
display(sub.head())
print("\nHead (terminal)")
display(ter.head())

# info() snapshots for report screenshot use
buf_sub = io.StringIO()
sub.info(buf=buf_sub)
print("\nsubmission info():")
print(buf_sub.getvalue())

buf_ter = io.StringIO()
ter.info(buf=buf_ter)
print("\nterminal info():")
print(buf_ter.getvalue())


## 3. Data quality checks (missingness, states)

In [ ]:
quality_cols = ["Submit", "Eligible", "Start", "End", "WaitTimeSec", "RunTimeSec", "Priority", "ReqTRES", "TimelimitRaw"]

rows = []
for col in quality_cols:
    series = get_col(ter, col)
    if pd.api.types.is_datetime64_any_dtype(series):
        missing = series.isna().sum()
    elif pd.api.types.is_numeric_dtype(series):
        missing = series.isna().sum()
    else:
        s = series.astype("string").str.strip()
        missing = (s.isna() | s.eq("") | s.str.lower().isin({"unknown", "none", "n/a"})).sum()
    total = len(ter)
    rows.append({"column": col, "missing_count": int(missing), "missing_rate": float(missing / total) if total else np.nan})

missing_df = pd.DataFrame(rows).sort_values("missing_rate", ascending=False)
print("Missingness summary (terminal):")
display(missing_df)

state = get_col(ter, "State").astype("string").str.strip().str.upper()
state = state.mask(state.isna() | state.eq(""), "MISSING")
state_counts = state.value_counts()
state_props = (state_counts / state_counts.sum()).rename("proportion")
state_table = pd.concat([state_counts.rename("count"), state_props], axis=1)
print("\nState distribution (count + proportion):")
display(state_table)

plt.figure(figsize=(9, 4))
(state_counts / state_counts.sum() * 100).sort_values(ascending=False).plot(kind="bar")
plt.ylabel("Percentage (%)")
plt.title("State distribution (terminal)")
save_current_fig("03_state_distribution")
plt.show()

wait_calc_ratio = safe_to_numeric(get_col(ter, "WaitTimeSec")).notna().mean()
run_calc_ratio = safe_to_numeric(get_col(ter, "RunTimeSec")).notna().mean()
print(f"\nWaitTimeSec computable ratio: {wait_calc_ratio:.2%}")
print(f"RunTimeSec computable ratio: {run_calc_ratio:.2%}")

METRICS["wait_calc_ratio"] = wait_calc_ratio
METRICS["run_calc_ratio"] = run_calc_ratio
METRICS["state_top"] = state_counts.index[0] if len(state_counts) else "N/A"
METRICS["state_top_ratio"] = float(state_counts.iloc[0] / state_counts.sum()) if len(state_counts) else np.nan


## 4. Submission behavior (weekday/hour/partition/QOS/resources)

In [ ]:
submit_ts = get_col(sub, "Submit")
if not pd.api.types.is_datetime64_any_dtype(submit_ts):
    submit_ts = safe_to_datetime(submit_ts)

sub_work = sub.copy()
sub_work["SubmitWeekday"] = submit_ts.dt.day_name()
sub_work["SubmitHour"] = submit_ts.dt.hour

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_counts = sub_work["SubmitWeekday"].value_counts().reindex(weekday_order).fillna(0)
weekday_prop = (weekday_counts / weekday_counts.sum()).fillna(0)

hour_counts = sub_work["SubmitHour"].value_counts().sort_index()
hour_prop = (hour_counts / hour_counts.sum()).fillna(0)

plt.figure(figsize=(9, 4))
(weekday_prop * 100).plot(kind="bar")
plt.ylabel("Percentage (%)")
plt.title("Submission proportion by weekday")
save_current_fig("04_submission_weekday_proportion")
plt.show()

plt.figure(figsize=(10, 4))
(hour_prop * 100).plot(kind="bar")
plt.ylabel("Percentage (%)")
plt.xlabel("Hour of day")
plt.title("Submission proportion by hour")
save_current_fig("05_submission_hour_proportion")
plt.show()

part_top = topn_with_other(get_col(sub_work, "Partition"), n=10)
qos_top = topn_with_other(get_col(sub_work, "QOS"), n=10)

plt.figure(figsize=(10, 4))
(part_top / part_top.sum() * 100).plot(kind="bar")
plt.ylabel("Percentage (%)")
plt.title("Partition distribution (Top 10 + Other)")
save_current_fig("06_partition_top10")
plt.show()

plt.figure(figsize=(10, 4))
(qos_top / qos_top.sum() * 100).plot(kind="bar")
plt.ylabel("Percentage (%)")
plt.title("QOS distribution (Top 10 + Other)")
save_current_fig("07_qos_top10")
plt.show()

reqcpus = safe_to_numeric(get_col(sub_work, "ReqCPUS"))
reqcpus = reqcpus[reqcpus.notna() & (reqcpus >= 0)]
if not reqcpus.empty:
    cap = reqcpus.quantile(0.99)
    plt.figure(figsize=(8, 4))
    plt.hist(reqcpus.clip(upper=cap), bins=60)
    plt.xlabel("ReqCPUS (clipped at 99th percentile)")
    plt.ylabel("Job count")
    plt.title("ReqCPUS distribution")
    save_current_fig("08_reqcpus_distribution")
    plt.show()

reqmem = safe_to_numeric(get_col(sub_work, "ReqMem_MB"))
reqmem = reqmem[reqmem.notna() & (reqmem >= 0)]
if not reqmem.empty:
    cap = reqmem.quantile(0.99)
    plt.figure(figsize=(8, 4))
    plt.hist(np.log10(reqmem.clip(lower=1e-6, upper=cap) + 1e-6), bins=60)
    plt.xlabel("log10(ReqMem_MB), clipped at 99th percentile")
    plt.ylabel("Job count")
    plt.title("ReqMem_MB distribution (log scale)")
    save_current_fig("09_reqmem_distribution_log")
    plt.show()

if "ReqTRES" in sub_work.columns:
    gpu_mask_sub = sub_work["ReqTRES"].astype("string").str.contains("gpu", case=False, na=False)
elif "ReqTRES" in ter.columns:
    gpu_mask_sub = ter["ReqTRES"].astype("string").str.contains("gpu", case=False, na=False)
else:
    gpu_mask_sub = pd.Series(False, index=sub_work.index)

gpu_ratio = float(gpu_mask_sub.mean()) if len(gpu_mask_sub) else np.nan
print(f"GPU job ratio: {gpu_ratio:.4%}")

METRICS["peak_weekday"] = weekday_prop.idxmax() if len(weekday_prop) else "N/A"
METRICS["peak_weekday_prop"] = float(weekday_prop.max()) if len(weekday_prop) else np.nan
METRICS["peak_hour"] = int(hour_prop.idxmax()) if len(hour_prop) and pd.notna(hour_prop.idxmax()) else "N/A"
METRICS["peak_hour_prop"] = float(hour_prop.max()) if len(hour_prop) else np.nan
METRICS["gpu_ratio"] = gpu_ratio


## 5. Scheduling outcomes (wait time, runtime, hold vs queue)

In [ ]:
# Wait time summary (hours)
wait_sec = safe_to_numeric(get_col(ter, "WaitTimeSec"))
if wait_sec.notna().sum() == 0 and {"Submit", "Start"}.issubset(set(ter.columns)):
    wait_sec = (ter["Start"] - ter["Submit"]).dt.total_seconds()

wait_stats = summary_quantiles_hours(wait_sec)
print("Wait time summary (hours):", wait_stats)

wait_hr = safe_to_numeric(wait_sec)
wait_hr = wait_hr[wait_hr.notna() & np.isfinite(wait_hr) & (wait_hr >= 0)] / 3600.0
if not wait_hr.empty:
    wcap = wait_hr.quantile(0.99)
    plt.figure(figsize=(8, 4))
    plt.hist(wait_hr.clip(upper=wcap), bins=60)
    plt.xlabel("Wait time (hours), clipped at 99th percentile")
    plt.ylabel("Job count")
    plt.title("Wait time distribution")
    save_current_fig("10_wait_time_distribution")
    plt.show()

    sorted_w = np.sort(wait_hr.values)
    ecdf_y = np.arange(1, len(sorted_w) + 1) / len(sorted_w)
    plt.figure(figsize=(8, 4))
    plt.plot(sorted_w, ecdf_y)
    plt.xlabel("Wait time (hours)")
    plt.ylabel("ECDF")
    plt.title("Wait time ECDF")
    save_current_fig("11_wait_time_ecdf")
    plt.show()

# Runtime summary (COMPLETED only)
state_upper = get_col(ter, "State").astype("string").str.upper().str.strip()
completed = ter[state_upper.eq("COMPLETED")].copy()
runtime_sec = safe_to_numeric(get_col(completed, "RunTimeSec"))
if runtime_sec.notna().sum() == 0 and {"Start", "End"}.issubset(set(completed.columns)):
    runtime_sec = (completed["End"] - completed["Start"]).dt.total_seconds()

runtime_stats = summary_quantiles_hours(runtime_sec)
print("Runtime summary (hours, COMPLETED only):", runtime_stats)

runtime_hr = safe_to_numeric(runtime_sec)
runtime_hr = runtime_hr[runtime_hr.notna() & np.isfinite(runtime_hr) & (runtime_hr >= 0)] / 3600.0
if not runtime_hr.empty:
    rcap = runtime_hr.quantile(0.99)
    plt.figure(figsize=(8, 4))
    plt.hist(runtime_hr.clip(upper=rcap), bins=60)
    plt.xlabel("Runtime (hours), clipped at 99th percentile")
    plt.ylabel("Job count")
    plt.title("Runtime distribution (COMPLETED)")
    save_current_fig("12_runtime_distribution_completed")
    plt.show()

# Hold and QueueWait decomposition (requires Eligible)
hold_stats = None
queue_stats = None
if {"Submit", "Eligible", "Start"}.issubset(set(ter.columns)):
    hold_sec = (ter["Eligible"] - ter["Submit"]).dt.total_seconds()
    queue_sec = (ter["Start"] - ter["Eligible"]).dt.total_seconds()

    hold_stats = summary_quantiles_hours(hold_sec)
    queue_stats = summary_quantiles_hours(queue_sec)

    print("Hold summary (Submit -> Eligible, hours):", hold_stats)
    print("QueueWait summary (Eligible -> Start, hours):", queue_stats)

    hold_hr = safe_to_numeric(hold_sec)
    hold_hr = hold_hr[hold_hr.notna() & np.isfinite(hold_hr) & (hold_hr >= 0)] / 3600.0

    queue_hr = safe_to_numeric(queue_sec)
    queue_hr = queue_hr[queue_hr.notna() & np.isfinite(queue_hr) & (queue_hr >= 0)] / 3600.0

    if (not hold_hr.empty) or (not queue_hr.empty):
        plt.figure(figsize=(8, 4))
        data = []
        labels = []
        if not hold_hr.empty:
            data.append(hold_hr.clip(upper=hold_hr.quantile(0.99)))
            labels.append("Hold: Submit->Eligible")
        if not queue_hr.empty:
            data.append(queue_hr.clip(upper=queue_hr.quantile(0.99)))
            labels.append("QueueWait: Eligible->Start")
        plt.boxplot(data, labels=labels, showfliers=False)
        plt.ylabel("Hours")
        plt.title("Hold vs QueueWait (clipped at 99th percentile)")
        save_current_fig("13_hold_vs_queuewait_boxplot")
        plt.show()

    hold_med = hold_stats["median"] if hold_stats else np.nan
    queue_med = queue_stats["median"] if queue_stats else np.nan
    if pd.notna(hold_med) and pd.notna(queue_med):
        dominant = "Submit->Eligible (Hold)" if hold_med > queue_med else "Eligible->Start (QueueWait)"
        print(f"Dominant median wait segment: {dominant}")
        METRICS["dominant_wait_segment"] = dominant
else:
    print("Eligible not available in terminal table; skipping hold/queue decomposition.")

METRICS["wait_stats"] = wait_stats
METRICS["runtime_stats"] = runtime_stats
METRICS["hold_stats"] = hold_stats
METRICS["queue_stats"] = queue_stats


## 6. Segment comparisons (GPU vs non-GPU / priority / partition)

In [ ]:
# A) GPU vs non-GPU
if "ReqTRES" in ter.columns:
    gpu_mask = ter["ReqTRES"].astype("string").str.contains("gpu", case=False, na=False)

    wait_gpu = summary_quantiles_hours(wait_sec[gpu_mask])
    wait_cpu = summary_quantiles_hours(wait_sec[~gpu_mask])

    if len(runtime_sec):
        runtime_gpu_series = runtime_sec[gpu_mask.loc[runtime_sec.index]]
        runtime_cpu_series = runtime_sec[(~gpu_mask).loc[runtime_sec.index]]
    else:
        runtime_gpu_series = pd.Series(dtype="float64")
        runtime_cpu_series = pd.Series(dtype="float64")

    run_gpu = summary_quantiles_hours(runtime_gpu_series)
    run_cpu = summary_quantiles_hours(runtime_cpu_series)

    print("GPU wait summary:", wait_gpu)
    print("non-GPU wait summary:", wait_cpu)
    print("GPU runtime summary:", run_gpu)
    print("non-GPU runtime summary:", run_cpu)

    state_gpu = get_col(ter[gpu_mask], "State").astype("string").str.upper().str.strip()
    state_cpu = get_col(ter[~gpu_mask], "State").astype("string").str.upper().str.strip()

    bad_states = {"FAILED", "TIMEOUT", "OOM", "OUT_OF_MEMORY"}
    gpu_bad_rate = state_gpu.isin(bad_states).mean() if len(state_gpu) else np.nan
    cpu_bad_rate = state_cpu.isin(bad_states).mean() if len(state_cpu) else np.nan
    print(f"GPU bad-state rate (FAILED/TIMEOUT/OOM): {gpu_bad_rate:.2%}")
    print(f"non-GPU bad-state rate: {cpu_bad_rate:.2%}")

    comp = pd.DataFrame({
        "group": ["GPU", "non-GPU"],
        "wait_p50_h": [wait_gpu["median"], wait_cpu["median"]],
        "wait_p90_h": [wait_gpu["p90"], wait_cpu["p90"]],
    })
    comp_plot = comp.set_index("group")
    plt.figure(figsize=(7, 4))
    comp_plot.plot(kind="bar", ax=plt.gca())
    plt.ylabel("Hours")
    plt.title("GPU vs non-GPU wait (p50 / p90)")
    save_current_fig("14_gpu_vs_nongpu_wait")
    plt.show()

    METRICS["gpu_wait_median"] = wait_gpu["median"]
    METRICS["nongpu_wait_median"] = wait_cpu["median"]
    METRICS["gpu_bad_rate"] = gpu_bad_rate
    METRICS["cpu_bad_rate"] = cpu_bad_rate
else:
    print("ReqTRES not available; skipping GPU vs non-GPU analysis.")

# B) Priority buckets (quartiles)
if "Priority" in ter.columns:
    pr = safe_to_numeric(ter["Priority"])
    valid = pr.notna()
    if valid.sum() > 100:
        tmp = ter.loc[valid, ["Priority"]].copy()
        tmp["Priority"] = pr.loc[valid]
        try:
            tmp["prio_bucket"] = pd.qcut(tmp["Priority"], 4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop")
            tmp["WaitH"] = safe_to_numeric(wait_sec.loc[tmp.index]) / 3600.0
            grp = tmp.groupby("prio_bucket", observed=False)["WaitH"]
            prio_summary = grp.quantile([0.5, 0.9]).unstack()
            prio_summary.columns = ["p50_h", "p90_h"]
            print("\nPriority bucket wait summary:")
            display(prio_summary)

            plt.figure(figsize=(8, 4))
            prio_summary.plot(kind="bar", ax=plt.gca())
            plt.ylabel("Hours")
            plt.title("Wait time by priority bucket")
            save_current_fig("15_priority_bucket_wait")
            plt.show()

            METRICS["priority_available"] = True
        except ValueError:
            print("Priority lacks enough unique values for qcut; skipping priority bucket analysis.")
            METRICS["priority_available"] = False
    else:
        print("Priority has too few non-null values; skipping priority bucket analysis.")
        METRICS["priority_available"] = False
else:
    print("Priority not available; skipping priority bucket analysis.")
    METRICS["priority_available"] = False

# C) Partition effect (Top 5)
if "Partition" in ter.columns:
    part = ter["Partition"].astype("string").str.strip().fillna("MISSING")
    top5 = part.value_counts().head(5).index.tolist()
    t2 = ter[part.isin(top5)].copy()
    t2["Partition"] = t2["Partition"].astype("string")
    t2["WaitH"] = safe_to_numeric(wait_sec.loc[t2.index]) / 3600.0
    t2["RunH"] = safe_to_numeric(get_col(t2, "RunTimeSec")) / 3600.0

    def _q(s, q):
        s = s.dropna()
        return np.nan if s.empty else float(np.nanquantile(s, q))

    part_summary = t2.groupby("Partition", observed=False).agg(
        wait_p50_h=("WaitH", lambda s: _q(s, 0.5)),
        wait_p90_h=("WaitH", lambda s: _q(s, 0.9)),
        run_p50_h=("RunH", lambda s: _q(s, 0.5)),
        jobs=("Partition", "size"),
    ).sort_values("jobs", ascending=False)

    print("\nPartition effect (Top 5 partitions):")
    display(part_summary)

    plt.figure(figsize=(9, 4))
    part_summary[["wait_p50_h", "wait_p90_h"]].plot(kind="bar", ax=plt.gca())
    plt.ylabel("Hours")
    plt.title("Wait time by top partitions")
    save_current_fig("16_partition_effect_wait")
    plt.show()


## 7. Key findings (numbers + saved plots)

In [ ]:
wait_stats = METRICS.get("wait_stats", {})
runtime_stats = METRICS.get("runtime_stats", {})
hold_stats = METRICS.get("hold_stats")
queue_stats = METRICS.get("queue_stats")

findings = []
findings.append(
    f"- Jobs are most frequently submitted on **{METRICS.get('peak_weekday', 'N/A')}** "
    f"({pct(METRICS.get('peak_weekday_prop'))}), and peak submission hour is **{METRICS.get('peak_hour', 'N/A')}** "
    f"({pct(METRICS.get('peak_hour_prop'))})."
)

findings.append(
    f"- Median wait time is **{wait_stats.get('median', np.nan):.2f} h**, "
    f"p90 is **{wait_stats.get('p90', np.nan):.2f} h**, "
    f"p95 is **{wait_stats.get('p95', np.nan):.2f} h**."
)

findings.append(
    f"- For COMPLETED jobs, median runtime is **{runtime_stats.get('median', np.nan):.2f} h**, "
    f"p90 is **{runtime_stats.get('p90', np.nan):.2f} h**."
)

findings.append(
    f"- GPU jobs account for **{pct(METRICS.get('gpu_ratio'))}** of jobs."
)

if hold_stats and queue_stats:
    findings.append(
        f"- Wait decomposition: Submit->Eligible median **{hold_stats.get('median', np.nan):.2f} h**, "
        f"Eligible->Start median **{queue_stats.get('median', np.nan):.2f} h**; dominant segment: "
        f"**{METRICS.get('dominant_wait_segment', 'N/A')}**."
    )

if pd.notna(METRICS.get("gpu_wait_median", np.nan)) and pd.notna(METRICS.get("nongpu_wait_median", np.nan)):
    denom = METRICS["nongpu_wait_median"]
    ratio = np.nan if (pd.isna(denom) or denom == 0) else METRICS["gpu_wait_median"] / denom
    findings.append(
        f"- GPU vs non-GPU median wait: **{METRICS['gpu_wait_median']:.2f} h** vs **{METRICS['nongpu_wait_median']:.2f} h** "
        f"(ratio: **{ratio:.2f}x**)."
    )

# Keep 3-6 bullets
findings = findings[:6]

report = [
    "# EDA Key Findings (Job-Level, 2025)",
    "",
    *findings,
    "",
    "## Saved figures",
]
report += [f"- `{p}`" for p in FIGURE_FILES]

FINDINGS_PATH.write_text("\n".join(report), encoding="utf-8")

print(f"Wrote findings file: {FINDINGS_PATH}")
print("\n".join(report))
